In [12]:
import kagglehub

path = kagglehub.dataset_download("ascanipek/skin-diseases")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/skin-diseases


In [15]:
import numpy as np
import os
import cv2
import matplotlib.pyplot as plt

from tensorflow.keras.applications import resnet50
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense

In [ ]:
import os
import cv2

train_data_path = '/root/.cache/kagglehub/datasets/ascanipek/skin-diseases/versions/3/kaggle/train'
val_data_path = '/root/.cache/kagglehub/datasets/ascanipek/skin-diseases/versions/3/kaggle/val'


train_data = []
val_data = []

for folder in os.listdir(train_data_path):
    folder_path = os.path.join(train_data_path, folder)
    if not os.path.isdir(folder_path):
        continue

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        train_data.append(file_path)


for folder in os.listdir(val_data_path):
    folder_path = os.path.join(val_data_path, folder)
    if not os.path.isdir(folder_path):
        continue

    for file in os.listdir(folder_path):
        file_path = os.path.join(folder_path, file)
        val_data.append(file_path)

print("Total training images:", len(train_data))
print("Total validation images:", len(val_data))

In [ ]:
from tensorflow.keras.layers import Dropout
from tensorflow.keras.models import Model

base_model = ResNet50(include_top=False, weights='imagenet', input_shape=(400, 400, 3))
base_model.trainable = False
num_classes = 6
x = GlobalAveragePooling2D()(base_model.output)
x = Dense(512, activation='relu')(x)
predictions = Dense(num_classes, activation='softmax')(x)
model = Model(inputs=base_model.input, outputs=predictions)
from tensorflow.keras.optimizers import Adam

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])


In [ ]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator

train_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

val_datagen = ImageDataGenerator(
    preprocessing_function=preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    train_data_path,
    target_size=(400, 400),
    batch_size=64,
    class_mode='categorical',
    shuffle=True
)

val_generator = val_datagen.flow_from_directory(
    val_data_path,
    target_size=(400, 400),
    batch_size=64,
    class_mode='categorical',
    shuffle=False
)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [21]:
from keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=5,
    mode='max',
    verbose=1
)

checkpoint = ModelCheckpoint(
    '/content/drive/MyDrive/kalitero_modelResNet50.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

class_indices = train_generator.class_indices
classes = list(class_indices.keys())

counts = [len(os.listdir(os.path.join(train_data_path, cls))) for cls in classes]

weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(len(classes)),
    y=np.concatenate([
        np.full(count, idx) for idx, count in enumerate(counts)
    ])
)
class_weights = dict(enumerate(weights))
print("Class Weights:", class_weights)

In [ ]:
history = model.fit(
    train_generator,
    steps_per_epoch=int(np.ceil(train_generator.samples / train_generator.batch_size)),  # Convert to int
    epochs=35,
    validation_data=val_generator,
    validation_steps=int(np.ceil(val_generator.samples / val_generator.batch_size)),
    class_weight=class_weights,
    callbacks=[early_stop, checkpoint]


)



In [ ]:
import matplotlib.pyplot as plt



plt.figure(figsize=(12, 6))

# Plot accuracy
plt.subplot(1, 2, 1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epochs')
plt.ylabel('Accuracy')
plt.legend(loc='lower right')

# Plot loss
plt.subplot(1, 2, 2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.legend(loc='upper right')

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

y_true = val_generator.classes

y_pred_prob = model.predict(val_generator)

y_pred = np.argmax(y_pred_prob, axis=1)

In [ ]:
class_labels = list(val_generator.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_labels))

In [ ]:
cm = confusion_matrix(y_true, y_pred)

# Plot confusion matrix
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=class_labels, yticklabels=class_labels)
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix")
plt.show()
